In [56]:
import sys
from pathlib import Path
import pandas as pd 
import numpy as np

sys.path.append(str(Path.cwd().parent))

from src.data_loading import load_all_stations
from src.cleaning import replace_missing_values


In [57]:
raw = load_all_stations()
print(f"Raw data shape: {raw.shape}")
print(f"Raw data columns: {raw.columns}")
print(f"Raw data first 3 rows:\n{raw.head(3)}")

Raw data shape: (61130, 28)
Raw data columns: Index(['STATION', 'DATE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'TEMP',
       'TEMP_ATTRIBUTES', 'DEWP', 'DEWP_ATTRIBUTES', 'SLP', 'SLP_ATTRIBUTES',
       'STP', 'STP_ATTRIBUTES', 'VISIB', 'VISIB_ATTRIBUTES', 'WDSP',
       'WDSP_ATTRIBUTES', 'MXSPD', 'GUST', 'MAX', 'MAX_ATTRIBUTES', 'MIN',
       'MIN_ATTRIBUTES', 'PRCP', 'PRCP_ATTRIBUTES', 'SNDP', 'FRSHTT'],
      dtype='str')
Raw data first 3 rows:
       STATION       DATE   LATITUDE  LONGITUDE  ELEVATION  \
0  41184099999 1996-01-01  25.613483  55.938817      31.08   
1  41184099999 1996-01-02  25.613483  55.938817      31.08   
2  41184099999 1996-01-03  25.613483  55.938817      31.08   

                               NAME  TEMP  TEMP_ATTRIBUTES  DEWP  \
0  RAS AL KHAIMAH INTERNATIONAL, AE  68.8               22  58.7   
1  RAS AL KHAIMAH INTERNATIONAL, AE  70.4               21  60.4   
2  RAS AL KHAIMAH INTERNATIONAL, AE  70.0               23  59.5   

   DEWP_ATTRIBUTE

In [58]:
print(f"Row count for each station:\n{raw['NAME'].value_counts()}")
print(f"Earliest date for each station:\n{raw.groupby('NAME')['DATE'].min()}")
print(f"Latest date for each station:\n{raw.groupby('NAME')['DATE'].max()}")

Row count for each station:
NAME
ABU DHABI INTERNATIONAL, AE         10219
DUBAI INTERNATIONAL, AE             10216
SHARJAH INTERNATIONAL, AE           10205
AL AIN INTERNATIONAL, AE            10192
FUJAIRAH INTERNATIONAL, AE          10161
RAS AL KHAIMAH INTERNATIONAL, AE    10137
Name: count, dtype: int64
Earliest date for each station:
NAME
ABU DHABI INTERNATIONAL, AE        1996-01-01
AL AIN INTERNATIONAL, AE           1996-01-01
DUBAI INTERNATIONAL, AE            1996-01-01
FUJAIRAH INTERNATIONAL, AE         1996-01-01
RAS AL KHAIMAH INTERNATIONAL, AE   1996-01-01
SHARJAH INTERNATIONAL, AE          1996-01-01
Name: DATE, dtype: datetime64[us]
Latest date for each station:
NAME
ABU DHABI INTERNATIONAL, AE        2023-12-31
AL AIN INTERNATIONAL, AE           2023-12-31
DUBAI INTERNATIONAL, AE            2023-12-31
FUJAIRAH INTERNATIONAL, AE         2023-12-31
RAS AL KHAIMAH INTERNATIONAL, AE   2023-12-31
SHARJAH INTERNATIONAL, AE          2023-12-31
Name: DATE, dtype: datetime64[u

In [59]:
print(f"Number of days between earliest and latest date for each station:\n{raw.groupby('NAME')['DATE'].apply(lambda x: (x.max() - x.min()).days)}")

Number of days between earliest and latest date for each station:
NAME
ABU DHABI INTERNATIONAL, AE         10226
AL AIN INTERNATIONAL, AE            10226
DUBAI INTERNATIONAL, AE             10226
FUJAIRAH INTERNATIONAL, AE          10226
RAS AL KHAIMAH INTERNATIONAL, AE    10226
SHARJAH INTERNATIONAL, AE           10226
Name: DATE, dtype: int64


In [60]:
print(f"missing days for each station:\n{raw.groupby('NAME')['DATE'].apply(lambda x: (x.max() - x.min()).days + 1 - len(x))}")
print("Total: ", raw.groupby('NAME')['DATE'].apply(lambda x: (x.max() - x.min()).days + 1 - len(x)).sum())

missing days for each station:
NAME
ABU DHABI INTERNATIONAL, AE          8
AL AIN INTERNATIONAL, AE            35
DUBAI INTERNATIONAL, AE             11
FUJAIRAH INTERNATIONAL, AE          66
RAS AL KHAIMAH INTERNATIONAL, AE    90
SHARJAH INTERNATIONAL, AE           22
Name: DATE, dtype: int64
Total:  232


In [61]:
dates = pd.date_range('1996-01-01', '2023-12-31', freq='D')
rak = raw[raw['STATION'] == 41184099999]
present = pd.DatetimeIndex(rak["DATE"])

missing_dates = dates.difference(present)
print(f"missing days from rak: {len(missing_dates)}")

missing_by_month = pd.Series(missing_dates).dt.to_period("M").value_counts().sort_index()
print(f"Missing by Month: \n{missing_by_month}")

missing days from rak: 90
Missing by Month: 
1996-08     2
1997-09     1
1997-10     9
1998-02     1
1998-03     1
1999-01     1
2005-12    25
2006-01    31
2006-02    13
2006-08     2
2020-04     1
2021-02     3
Freq: M, Name: count, dtype: int64
